In [ ]:
# ============================================================
# OVERNIGHT PILOTS v4 — SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/sebastianquispearias/tesis-seg.git /content/tesis-seg 2>/dev/null || echo 'Already cloned'
!pip install -q -r /content/tesis-seg/requirements.txt

import torch, sys, os, time, json, platform, importlib.metadata
print('Python:', sys.version)
print('torch:', torch.__version__)
print('CUDA:', torch.version.cuda)
!cd /content/tesis-seg && git log --oneline -1
!pip show albumentations | grep Version

sys.path.insert(0, '/content/tesis-seg')

# --- Patch cloned datasets.py: enable cfg["batch_size_unlab"] override ---
# DeepLabV3+ ASPP has a global-pool->BatchNorm branch that crashes when the
# unlabeled batch is 1 (default batch_size//4). This lets the DeepLabV3+ cells
# set cfg["batch_size_unlab"]=2. No-op for every other run (default unchanged).
_ds_path = '/content/tesis-seg/src/datasets.py'
_ds_src = open(_ds_path).read()
_old = 'batch_size_unlab = max(1, cfg["batch_size"] // 4)'
_new = 'batch_size_unlab = cfg.get("batch_size_unlab") or max(1, cfg["batch_size"] // 4)'
if _new in _ds_src:
    print('[PATCH] datasets.py already has batch_size_unlab override (from GitHub)')
elif _old in _ds_src:
    open(_ds_path, 'w').write(_ds_src.replace(_old, _new))
    print('[PATCH] datasets.py patched: batch_size_unlab override enabled')
else:
    print('[PATCH][WARN] target line not found in datasets.py - check manually')

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
import segmentation_models_pytorch as smp
import src.models as models_module
import src.train as train_module

_orig_create = models_module.create_model
_create_model_call_log = []  # tracks every call

def _patch_create_model(fn):
    models_module.create_model = fn
    train_module.create_model = fn
    print(f'[PATCH] models_module.create_model = {fn.__name__} (id={id(fn)})')
    print(f'[PATCH] train_module.create_model  = {fn.__name__} (id={id(fn)})')
    print(f'[VERIFY] models_module id: {id(models_module.create_model)}')
    print(f'[VERIFY] train_module id:  {id(train_module.create_model)}')
    print(f'[VERIFY] patched fn id:    {id(fn)}')
    assert models_module.create_model is fn, 'PATCH FAILED on models_module!'
    assert train_module.create_model is fn, 'PATCH FAILED on train_module!'
    print('[VERIFY] Both modules patched and verified OK.')

def _restore_create_model():
    _patch_create_model(_orig_create)

# ── Fingerprint helpers ──
def _fp_get_version(pkg):
    try: return importlib.metadata.version(pkg)
    except: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump({'pre_run': pre, 'post_run': post}, f, indent=2, default=str)

def _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds, unlabeled_ds=None, temporal_unlab_ds=None):
    try: gpu_name = torch.cuda.get_device_name(0)
    except: gpu_name = 'n/a'
    _m = models_module.create_model(cfg['arch'], cfg['backbone'], cfg['n_classes'])
    mf = {'total_params': sum(p.numel() for p in _m.parameters()),
          'trainable_params': sum(p.numel() for p in _m.parameters() if p.requires_grad)}
    del _m
    return {
        'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'environment': {
            'python': sys.version, 'torch': torch.__version__,
            'cuda': torch.version.cuda,
            'segmentation_models_pytorch': _fp_get_version('segmentation-models-pytorch'),
            'albumentations': _fp_get_version('albumentations'),
            'platform': platform.platform(), 'gpu_name': gpu_name,
        },
        'effective_cfg': cfg,
        'dataset_facts': {
            'len_train_ds': len(train_ds), 'len_val_ds': len(val_ds),
            'len_test_ds': len(test_ds),
            'len_unlabeled_ds': len(unlabeled_ds) if unlabeled_ds else None,
        },
        'model_fingerprint': mf,
    }

def _fp_collect_post(artifacts, results, call_log, expected_weights):
    return {
        'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'finished_successfully': True,
        'best_path': artifacts.get('best_path'),
        'test_metrics': results.get('test_metrics') if results else None,
        'model_init_verification': {
            'expected_encoder_weights': expected_weights,
            'create_model_called_count': len(call_log),
            'calls': call_log,
            'models_module_patched': True,
            'train_module_patched': True,
        },
    }

print('Setup OK')


# Part A: From-Scratch (6 runs pendientes)
UNet++ sin ImageNet. 3 seeds UNM + 3 seeds INCA.
Baseline UNM ImageNet: seed_0=0.8070, seed_1=0.8158, seed_2=0.7786
Baseline INCA ImageNet: seed_0=0.9061, seed_1=0.9082, seed_2=0.9027

In [ ]:
# === RUN 1/9: supervised_fromscratch/seed_0 (FROM-SCRATCH) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = None
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "supervised_fromscratch"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'null'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNM {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.807')
        print(f'  Delta: {f1 - 0.807:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 2/9: supervised_fromscratch/seed_1 (FROM-SCRATCH) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = None
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "supervised_fromscratch"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'null'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNM {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.8158')
        print(f'  Delta: {f1 - 0.8158:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 3/9: supervised_fromscratch/seed_2 (FROM-SCRATCH) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = None
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "supervised_fromscratch"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'null'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNM {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.7786')
        print(f'  Delta: {f1 - 0.7786:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 4/9: supervised_inca_fromscratch/seed_0 (FROM-SCRATCH) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = None
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "inca"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "supervised_inca_fromscratch"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = False

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'null'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'INCA {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.9061')
        print(f'  Delta: {f1 - 0.9061:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 5/9: supervised_inca_fromscratch/seed_1 (FROM-SCRATCH) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = None
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "inca"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "supervised_inca_fromscratch"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = False

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'null'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'INCA {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.9082')
        print(f'  Delta: {f1 - 0.9082:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 6/9: supervised_inca_fromscratch/seed_2 (FROM-SCRATCH) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = None
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "inca"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "supervised_inca_fromscratch"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = False

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'null'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'INCA {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.9027')
        print(f'  Delta: {f1 - 0.9027:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


# Part B: SSL on Other Architectures (3 seeds each)
U-Net, FPN, DeepLabV3+ with Mean Teacher std_matched r15. UNM.
Baselines supervised (runs_final_v1):
- U-Net: seed_0=0.8528, seed_1=0.8452, seed_2=0.7873
- FPN: seed_0=0.7992, seed_1=0.8056, seed_2=0.8178
- DeepLabV3+: seed_0=0.7394, seed_1=0.7522, seed_2=0.7478

In [ ]:
# === RUN 7/9: mean_teacher_unet_std_matched_r15/seed_0 (UNET + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_unet_std_matched_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unet"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNM {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.8528')
        print(f'  Delta: {f1 - 0.8528:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 7/9: mean_teacher_fpn_std_matched_r15/seed_0 (FPN + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_fpn_std_matched_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "fpn"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNM {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.7992')
        print(f'  Delta: {f1 - 0.7992:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === RUN 7/9: mean_teacher_deeplabv3plus_std_matched_r15/seed_0 (DEEPLABV3PLUS + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []  # reset for this run

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_deeplabv3plus_std_matched_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "deeplabv3plus"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["batch_size_unlab"] = 2  # DeepLabV3+ ASPP BatchNorm needs unlabeled batch >1 (was //4=1)
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNM {_EXP_NAME} seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Baseline: F1=0.7394')
        print(f'  Delta: {f1 - 0.7394:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === mean_teacher_unet_std_matched_r15/seed_1 (UNET + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_unet_std_matched_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unet"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNET MT-r15 seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Supervised baseline: F1=0.8452')
        print(f'  Delta: {f1 - 0.8452:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === mean_teacher_unet_std_matched_r15/seed_2 (UNET + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_unet_std_matched_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "unet"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'UNET MT-r15 seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Supervised baseline: F1=0.7873')
        print(f'  Delta: {f1 - 0.7873:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === mean_teacher_fpn_std_matched_r15/seed_1 (FPN + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_fpn_std_matched_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "fpn"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'FPN MT-r15 seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Supervised baseline: F1=0.8056')
        print(f'  Delta: {f1 - 0.8056:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === mean_teacher_fpn_std_matched_r15/seed_2 (FPN + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_fpn_std_matched_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "fpn"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'FPN MT-r15 seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Supervised baseline: F1=0.8178')
        print(f'  Delta: {f1 - 0.8178:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === mean_teacher_deeplabv3plus_std_matched_r15/seed_1 (DEEPLABV3PLUS + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_deeplabv3plus_std_matched_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "deeplabv3plus"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["batch_size_unlab"] = 2  # DeepLabV3+ ASPP BatchNorm needs unlabeled batch >1 (was //4=1)
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'DEEPLABV3PLUS MT-r15 seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Supervised baseline: F1=0.7522')
        print(f'  Delta: {f1 - 0.7522:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === mean_teacher_deeplabv3plus_std_matched_r15/seed_2 (DEEPLABV3PLUS + MT r15) ===
import gc; gc.collect(); torch.cuda.empty_cache()
_create_model_call_log = []

def _create_this_run(arch, backbone, n_classes=1, pretrained=False):
    _w = 'imagenet'
    _arch = arch.lower()
    _builders = {
        'unetpp': smp.UnetPlusPlus, 'unet': smp.Unet,
        'fpn': smp.FPN, 'deeplabv3plus': smp.DeepLabV3Plus,
    }
    if _arch in _builders:
        model = _builders[_arch](encoder_name=backbone, encoder_weights=_w,
                                in_channels=3, classes=n_classes)
        fc = list(model.encoder.parameters())[0]
        _std = fc.data.std().item()
        print(f'>>> CREATE_MODEL_CALLED_BY_TRAINING '
              f'arch={_arch} backbone={backbone} '
              f'encoder_weights={_w} in_channels=3 '
              f'first_conv_std={_std:.4f}')
        _create_model_call_log.append({
            'arch': _arch, 'backbone': backbone,
            'encoder_weights': str(_w), 'first_conv_std': round(_std, 4)
        })
        return model
    return _orig_create(arch, backbone, n_classes, pretrained)
_patch_create_model(_create_this_run)

cfg = get_default_config()
cfg["dataset"]   = "unm"
cfg["img_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]  = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
_EXP_NAME = "mean_teacher_deeplabv3plus_std_matched_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"
cfg["arch"]      = "deeplabv3plus"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1
cfg["seed"]      = _SEED
cfg["use_semi"]  = True
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
cfg["use_temp_consistency"] = False
cfg["lambda_t"]  = 0.0
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False
cfg["batch_size"]     = 5
cfg["batch_size_unlab"] = 2  # DeepLabV3+ ASPP BatchNorm needs unlabeled batch >1 (was //4=1)
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg['exp_dir']
_skip = os.path.isfile(os.path.join(_exp_dir, 'best_model.pt')) and os.path.isfile(os.path.join(_exp_dir, 'test_metrics.csv'))
if _skip:
    print(f'SKIP {_EXP_NAME}/seed_{_SEED}: already complete')
else:
    print(summarize_config(cfg))
    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
    loaders = build_dataloaders(cfg, train_ds, val_ds, test_ds,
                               unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_pre = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                              unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)
    _fp_path = os.path.join(cfg['exp_dir'], 'debug_fingerprint.json')
    _fp_save(_fp_pre, None, _fp_path)
    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts['model'], loaders,
                                     artifacts['best_path'], artifacts['history'])
        _fp_save(_fp_pre, _fp_collect_post(artifacts, results, _create_model_call_log, 'imagenet'), _fp_path)
        f1 = results['test_metrics']['f1_mean']
        print(f'\n{"="*50}')
        print(f'DEEPLABV3PLUS MT-r15 seed_{_SEED}')
        print(f'  F1: {f1:.4f}')
        print(f'  Supervised baseline: F1=0.7478')
        print(f'  Delta: {f1 - 0.7478:+.4f}')
        print(f'  create_model calls: {len(_create_model_call_log)}')
        for _c in _create_model_call_log:
            print(f'    {_c}')
        print('=' * 50)
    except Exception as e:
        import traceback; traceback.print_exc()
        _fp_save(_fp_pre, {'finished_successfully': False, 'exception': traceback.format_exc()}, _fp_path)


In [ ]:
# === ALL DONE ===
import time
print('\n' + '=' * 60)
print('ALL OVERNIGHT PILOTS COMPLETE')
print(time.strftime('%Y-%m-%d %H:%M:%S'))
print('=' * 60)

# Uncomment for overnight:
from google.colab import runtime; runtime.unassign()
